In [0]:
# ============================================
# SILVER LAYER — Clean, validate, enrich data
# ============================================

from pyspark.sql.functions import (
    col, when, isnan, isnull, count,
    round as spark_round, current_timestamp
)
from pyspark.sql.types import DoubleType

# ── 1. Load Bronze ───────────────────────────────────────────────
df_bronze = spark.table("fraud_db.bronze_transactions")
print(f"✅ Bronze rows loaded: {df_bronze.count()}")

# ── 2. Check nulls ───────────────────────────────────────────────
print("\n=== NULL CHECK ===")
null_counts = df_bronze.select([
    count(when(isnull(c) | isnan(c), c)).alias(c)
    for c in df_bronze.columns
    if c not in [
        "ingestion_timestamp",
        "source_file",
        "pipeline_version"
    ]
])
null_counts.show()

# ── 3. Check duplicates ──────────────────────────────────────────
total    = df_bronze.count()
distinct = df_bronze.dropDuplicates().count()
print(f"Duplicate rows found: {total - distinct}")

# ── 4. Clean & validate ──────────────────────────────────────────
df_silver = df_bronze \
    .dropDuplicates() \
    .filter(col("Amount") > 0) \
    .filter(col("Amount").isNotNull()) \
    .filter(col("Class").isNotNull()) \
    .withColumn("Amount", col("Amount").cast(DoubleType())) \
    .withColumn("Time",   col("Time").cast(DoubleType()))

# ── 5. Enrich with new features ──────────────────────────────────
df_silver = df_silver \
    .withColumn(
        "amount_category",
        when(col("Amount") < 10,   "micro")
       .when(col("Amount") < 100,  "small")
       .when(col("Amount") < 1000, "medium")
       .otherwise("large")
    ) \
    .withColumn(
        "is_high_value",
        when(col("Amount") > 1000, 1).otherwise(0)
    ) \
    .withColumn(
        "hour_of_day",
        (col("Time") / 3600 % 24).cast("integer")
    ) \
    .withColumn(
        "is_night",
        when(
            (col("hour_of_day") >= 22) |
            (col("hour_of_day") <= 4),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "amount_rounded",
        spark_round(col("Amount"), 2)
    ) \
    .withColumn("silver_timestamp", current_timestamp())

# ── 6. Preview enriched data ─────────────────────────────────────
print("\n=== SILVER PREVIEW ===")
df_silver.select(
    "Time", "Amount", "amount_category",
    "is_high_value", "hour_of_day",
    "is_night", "Class"
).show(10)

# ── 7. Class distribution ────────────────────────────────────────
print("\n=== CLASS DISTRIBUTION ===")
df_silver.groupBy("Class").count().show()

print("\n=== FRAUD BY AMOUNT CATEGORY ===")
df_silver.groupBy("amount_category", "Class") \
    .count() \
    .orderBy("amount_category", "Class") \
    .show()

print("\n=== FRAUD BY HOUR ===")
df_silver.groupBy("hour_of_day", "Class") \
    .count() \
    .orderBy("hour_of_day") \
    .show(48)

# ── 8. Write Silver Delta table ──────────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.silver_transactions")

print(f"\n✅ Silver table saved!")
print(f"   Rows written: {df_silver.count()}")
print(f"   Table: fraud_db.silver_transactions")


✅ Bronze rows loaded: 284807

=== NULL CHECK ===
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+

Duplicate rows found: 1081

=== SILVER PREVIEW ===
+-----+------+---------------+-------------+-----------+--------+-----+
| Time|Amount|amount_category|is_high_value|hour_of_day|is_night|Class|
+-----+------+---------------+-------------+-----------+--------+-----+
|160.0|  20.0|         